In [ ]:
# ============================================================
# 06_Chronos.ipynb
# Section 1 — Setup, Installation, Variant Verification
# Purpose: Install chronos-forecasting; verify which variant
#          (Plan A: Chronos-2 multivariate / Plan B: Chronos-T5
#          univariate with .embed()) runs on the T4; load data.
# ============================================================

# --- 1a. Install Chronos (official Amazon package) ---
!pip install -q chronos-forecasting

# --- 1b. Mount Drive + canonical paths ---

import os, sys, time
PROJECT_ROOT = os.path.join(DATA_DIR, "..")
DATA_DIR      = os.path.join(PROJECT_ROOT, "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
FIGURES_DIR   = os.path.join(PROJECT_ROOT, "figures")
SRC_DIR       = os.path.join(PROJECT_ROOT, "src")
sys.path.insert(0, SRC_DIR)

# --- 1c. Imports + GPU check ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error
from preprocessing import FINAL_FEATURES_FD001

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

import chronos
print(f"\nchronos-forecasting version: {chronos.__version__ if hasattr(chronos, '__version__') else 'unknown'}")
print(f"Available in chronos package: "
      f"{[n for n in dir(chronos) if 'Pipeline' in n or 'Chronos' in n]}")

# --- 1d. Plan A attempt: Chronos-2 (native multivariate) ---
plan_a_works = False
try:
    from chronos import Chronos2Pipeline
    print("\n[Plan A] Chronos2Pipeline found — attempting load...")
    t0 = time.time()
    pipe2 = Chronos2Pipeline.from_pretrained("amazon/chronos-2",
                                             device_map=DEVICE)
    print(f"[Plan A] Loaded in {time.time()-t0:.0f}s")
    print(f"[Plan A] Methods: "
          f"{[m for m in dir(pipe2) if not m.startswith('_')][:20]}")
    plan_a_works = True
except Exception as e:
    print(f"\n[Plan A] Not available: {type(e).__name__}: {e}")

# --- 1e. Plan B attempt: Chronos-T5-small with documented .embed() ---
plan_b_works = False
try:
    from chronos import ChronosPipeline
    print("\n[Plan B] Loading chronos-t5-small...")
    t0 = time.time()
    pipe1 = ChronosPipeline.from_pretrained("amazon/chronos-t5-small",
                                            device_map=DEVICE,
                                            torch_dtype=torch.bfloat16)
    print(f"[Plan B] Loaded in {time.time()-t0:.0f}s")

    # Tiny embedding test: one fake univariate series of length 50
    test_series = torch.tensor(np.linspace(0, 1, 50), dtype=torch.float32)
    t0 = time.time()
    emb, tok_state = pipe1.embed(test_series)
    print(f"[Plan B] .embed() works — output shape: {tuple(emb.shape)} "
          f"({time.time()-t0:.2f}s)")
    plan_b_works = True
except Exception as e:
    print(f"\n[Plan B] Failed: {type(e).__name__}: {e}")

print("\n" + "=" * 50)
print(f"Plan A (Chronos-2, multivariate): {'✓ AVAILABLE' if plan_a_works else '✗ not available'}")
print(f"Plan B (Chronos-T5, univariate):  {'✓ AVAILABLE' if plan_b_works else '✗ not available'}")
print("=" * 50)

# --- 1f. Load data + assertions (same as all notebooks) ---
train = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_full = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))

N_FEATURES = len(FINAL_FEATURES_FD001)
assert train.shape[0] == 20631
assert test_full.shape[0] == 13096
assert test_last.shape[0] == 100
assert all(f in train.columns for f in FINAL_FEATURES_FD001)

print(f"\nTrain {train.shape}, test-full {test_full.shape}, test-last {test_last.shape}")
print("✓ Data loaded. All checks passed.")

In [ ]:
# ============================================================
# NB06 — Section 1b: Chronos-2 backbone parameter count
# (for the Kolloquium — Herr Karri asks about model size)
# Backbone is pretrained and FULLY FROZEN; only a Ridge head
# (769 params) is trained on top for the fair full-history setup.
# ============================================================
import torch.nn as nn

# Try the likely attribute names for the underlying nn.Module
candidate = None
used = None
for attr in ["model", "inner_model"]:
    obj = getattr(pipe2, attr, None)
    if isinstance(obj, nn.Module):
        candidate = obj; used = f"pipe2.{attr}"; break
    # sometimes the module is one level deeper (e.g. pipe2.model.model)
    if obj is not None:
        inner = getattr(obj, "model", None)
        if isinstance(inner, nn.Module):
            candidate = inner; used = f"pipe2.{attr}.model"; break

if candidate is not None:
    n_total = sum(p.numel() for p in candidate.parameters())
    n_train = sum(p.numel() for p in candidate.parameters() if p.requires_grad)
    print(f"Chronos-2 backbone ({used}):")
    print(f"  Total parameters:     {n_total:,}")
    print(f"  Trainable parameters: {n_train:,}  (0 expected — frozen)")
else:
    print("Could not locate the nn.Module automatically. Attributes on pipe2:")
    print([a for a in dir(pipe2) if not a.startswith('_')])
    print("\nTell me which attribute holds the model and I'll adjust.")

print("\nTrained on top (fair full-history): Ridge head = 768 + 1 = 769 params")

In [ ]:
# ============================================================
# 06_Chronos.ipynb
# Section 2 — Metrics + Embedding Probe + Extraction Pipeline
# Purpose: (a) metrics identical to NB03-05; (b) probe the
#          Chronos-2 .embed() API on multivariate input;
#          (c) batched extraction function, verified on a sample.
# API note (from probe): Chronos-2 .embed() expects a 3-d tensor
#          (n_series, n_variates, history_length), e.g. (n, 13, 50).
# ============================================================

# --- 2a. Metrics (identical to Notebooks 03-05) ---
def nasa_score(y_true, y_pred):
    """PHM08 asymmetric scoring function (Saxena et al. 2008)."""
    d = y_pred - y_true
    scores = np.where(d < 0,
                      np.exp(-d / 13.0) - 1.0,
                      np.exp( d / 10.0) - 1.0)
    return float(np.sum(scores))

def evaluate_model(y_true, y_pred, model_name="Model", label_name="label"):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    ns   = nasa_score(y_true, y_pred)
    print(f"--- {model_name} | {label_name} ---")
    print(f"  MAE:        {mae:.2f} cycles")
    print(f"  RMSE:       {rmse:.2f} cycles")
    print(f"  NASA Score: {ns:.2f}")
    print()
    return {"model": model_name, "rul_label": label_name,
            "mae": round(mae, 2), "rmse": round(rmse, 2),
            "nasa_score": round(ns, 2)}

assert nasa_score(np.array([100.]), np.array([100.])) == 0.0
assert nasa_score(np.array([100.]), np.array([110.])) > nasa_score(np.array([100.]), np.array([90.]))
print("✓ Metrics defined and verified.\n")

# --- 2b. Verify the correct 3D input format: (n_series, n_variates, history) ---
print("=== Verifying Chronos-2 .embed() with correct 3D input (1, 13, 50) ===\n")

# One real window: engine 1, its last 50 cycles, all 13 sensors
e1 = train[train["unit"] == 1].sort_values("cycle")
window_np = e1[FINAL_FEATURES_FD001].values[-50:]        # (50, 13)
print(f"Probe window: {window_np.shape} (W=50 cycles, 13 sensors)")

x3d = torch.tensor(window_np.T[None, :, :], dtype=torch.float32)  # (1, 13, 50)
out = pipe2.embed(x3d)
emb_raw = out[0] if isinstance(out, tuple) else out
if isinstance(emb_raw, list):
    emb_raw = emb_raw[0]
print(f"Input (1, 13, 50) -> embedding output shape: "
      f"{tuple(emb_raw.shape) if hasattr(emb_raw, 'shape') else type(emb_raw)}")

# --- 2c. Pooling: mean-pool all axes except the last (feature) axis ---
def pool_embedding(emb):
    """Tensor of shape (..., D) -> 1D vector of dim D via mean-pooling
    over all leading axes (tokens, variates, etc.)."""
    e = emb.detach().float().cpu().numpy()
    e = np.squeeze(e)
    if e.ndim == 1:
        return e
    return e.reshape(-1, e.shape[-1]).mean(axis=0)

vec = pool_embedding(emb_raw)
EMB_DIM = vec.shape[0]
print(f"✓ Pooled embedding vector dim: {EMB_DIM}")

# --- 2d. Batched extraction using Chronos-2's internal batching ---
def extract_embeddings(windows, batch_size=64, desc=""):
    """windows: (n, W, 13) -> (n, EMB_DIM).
    Transposes to Chronos-2's (n_series, n_variates, history) format,
    uses .embed's internal batching, mean-pools per series."""
    x = torch.tensor(windows.transpose(0, 2, 1), dtype=torch.float32)  # (n,13,W)
    t0 = time.time()
    out = pipe2.embed(x, batch_size=batch_size)
    emb = out[0] if isinstance(out, tuple) else out
    if isinstance(emb, list):
        pooled = np.stack([pool_embedding(e) for e in emb])
    else:
        e = emb.detach().float().cpu().numpy()          # (n, ..., D)
        pooled = e.reshape(e.shape[0], -1, e.shape[-1]).mean(axis=1)
    dt = time.time() - t0
    print(f"  [{desc}] {len(windows)} windows in {dt:.1f}s "
          f"({dt/len(windows)*1000:.1f} ms/window)")
    return pooled.astype(np.float32)

# --- 2e. Window builder (same logic as Notebook 05) ---
def build_windows_from_df(df, window):
    """All sliding windows (one per row), zero-pre-padded.
    Returns X (n, window, 13), row order = (unit asc, cycle asc)."""
    X_list = []
    for unit, grp in df.groupby("unit"):
        grp = grp.sort_values("cycle")
        feats = grp[FINAL_FEATURES_FD001].values
        L = len(grp)
        for end in range(L):
            start = max(0, end - window + 1)
            seq = feats[start:end + 1]
            if len(seq) < window:
                seq = np.vstack([np.zeros((window - len(seq), N_FEATURES)),
                                 seq])
            X_list.append(seq)
    return np.array(X_list, dtype=np.float32)

# --- 2f. Verification on a small sample (engines 1-2) ---
sample_windows = build_windows_from_df(train[train["unit"] <= 2], window=50)
print(f"\nSample: {sample_windows.shape[0]} windows from engines 1-2")
sample_emb = extract_embeddings(sample_windows, desc="sample")
print(f"Sample embeddings: {sample_emb.shape}")
assert sample_emb.shape[1] == EMB_DIM
assert not np.isnan(sample_emb).any(), "NaNs in embeddings!"
print("\n✓ Section 2 complete: batched extraction pipeline verified.")

In [ ]:
# ============================================================
# 06_Chronos.ipynb
# Section 3 — Extraction + Validation Split + Sweep + Final Eval
# Protocol identical to NB04/05: same val engines (seed 42),
# same truncation (seed 123), 6-config sweep, single test eval.
# Backbone FROZEN throughout; only the regression head trains.
# ============================================================
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf
keras.utils.set_random_seed(42)
SEED = 42

# --- 3a. Validation split (IDENTICAL to NB04/05) ---
rng = np.random.RandomState(42)
all_units = train["unit"].unique()
val_units = rng.choice(all_units, size=20, replace=False)
train_units = np.array([u for u in all_units if u not in val_units])
train_sub = train[train["unit"].isin(train_units)]
val_sub   = train[train["unit"].isin(val_units)]
print(f"Training engines: {len(train_units)}, validation engines: {len(val_units)}")
print(f"Validation units: {sorted(val_units.tolist())}\n")

def make_truncation_points(val_sub, rng, min_keep=30):
    cuts = {}
    for unit, grp in val_sub.groupby("unit"):
        c_max = grp["cycle"].max()
        cuts[unit] = (rng.randint(min_keep, c_max), c_max)
    return cuts

cuts = make_truncation_points(val_sub, np.random.RandomState(123))
val_units_sorted = sorted(cuts)
y_val_true = np.array([cuts[u][1] - cuts[u][0] for u in val_units_sorted],
                      dtype=float)

def build_val_windows(val_sub, cuts, window, units_order):
    X_list = []
    for unit in units_order:
        cut, _ = cuts[unit]
        grp = val_sub[(val_sub["unit"] == unit) &
                      (val_sub["cycle"] <= cut)].sort_values("cycle")
        feats = grp[FINAL_FEATURES_FD001].values
        seq = feats[-window:]
        if len(seq) < window:
            seq = np.vstack([np.zeros((window - len(seq), N_FEATURES)), seq])
        X_list.append(seq)
    return np.array(X_list, dtype=np.float32)

def build_last_window_sequences(df_full, window, units_order):
    X_list = []
    for unit in units_order:
        grp = df_full[df_full["unit"] == unit].sort_values("cycle")
        feats = grp[FINAL_FEATURES_FD001].values
        seq = feats[-window:]
        if len(seq) < window:
            seq = np.vstack([np.zeros((window - len(seq), N_FEATURES)), seq])
        X_list.append(seq)
    return np.array(X_list, dtype=np.float32)

# --- 3b. Extract embeddings for all context lengths (the big step) ---
CONTEXTS = [30, 50, 100]
emb_cache = {}          # context -> dict with train/val/test embeddings
extract_times = {}

y_true = test_last["RUL_true"].values
units_test = test_last["unit"].values

for ctx in CONTEXTS:
    print(f"=== Context length {ctx} ===")
    t0 = time.time()
    Xw_tr  = build_windows_from_df(train_sub, ctx)
    Xw_trF = build_windows_from_df(train, ctx)          # full train (final runs)
    Xw_val = build_val_windows(val_sub, cuts, ctx, val_units_sorted)
    Xw_te  = build_last_window_sequences(test_full, ctx, units_test)
    E_tr  = extract_embeddings(Xw_tr,  desc=f"train80-ctx{ctx}")
    E_trF = extract_embeddings(Xw_trF, desc=f"train100-ctx{ctx}")
    E_val = extract_embeddings(Xw_val, desc=f"val-ctx{ctx}")
    E_te  = extract_embeddings(Xw_te,  desc=f"test-ctx{ctx}")
    extract_times[ctx] = time.time() - t0
    emb_cache[ctx] = dict(E_tr=E_tr, E_trF=E_trF, E_val=E_val, E_te=E_te)
    print(f"  total for ctx={ctx}: {extract_times[ctx]/60:.1f} min\n")
    del Xw_tr, Xw_trF, Xw_val, Xw_te

# --- 3c. Regression heads + sweep ---
def build_head(input_dim, arch):
    layers_list = [layers.Input(shape=(input_dim,))]
    for n in arch:
        layers_list += [layers.Dense(n, activation="relu"),
                        layers.Dropout(0.2)]
    layers_list += [layers.Dense(1)]
    m = keras.Sequential(layers_list)
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss="mse")
    return m

SWEEP = [
    {"ctx": 30,  "arch": (64,)},
    {"ctx": 50,  "arch": (64,)},
    {"ctx": 100, "arch": (64,)},
    {"ctx": 30,  "arch": (128, 64)},
    {"ctx": 50,  "arch": (128, 64)},
    {"ctx": 100, "arch": (128, 64)},
]
EPOCHS, BATCH, PATIENCE = 100, 64, 10

def get_labels(df, label_col):
    """Labels aligned with build_windows_from_df row order."""
    return df.sort_values(["unit", "cycle"])[label_col].values.astype(np.float32)

def run_head_sweep(label_col):
    y_max = float(train_sub[label_col].max())
    print(f"  (label scaling: y / {y_max:.0f})")
    y_tr = get_labels(train_sub, label_col)
    results = []
    for cfg in SWEEP:
        ctx, arch = cfg["ctx"], cfg["arch"]
        E_tr, E_val = emb_cache[ctx]["E_tr"], emb_cache[ctx]["E_val"]
        keras.utils.set_random_seed(SEED)
        head = build_head(E_tr.shape[1], arch)
        es = keras.callbacks.EarlyStopping(monitor="val_loss",
                                           patience=PATIENCE,
                                           restore_best_weights=True)
        t0 = time.time()
        h = head.fit(E_tr, y_tr / y_max, validation_split=0.1,
                     epochs=EPOCHS, batch_size=BATCH,
                     callbacks=[es], verbose=0)
        t_train = time.time() - t0
        pred_v = head.predict(E_val, verbose=0).ravel() * y_max
        rmse_v = np.sqrt(mean_squared_error(y_val_true, pred_v))
        results.append({**cfg, "val_rmse": rmse_v,
                        "epochs_run": len(h.history["loss"]),
                        "head_train_s": t_train})
        print(f"  ctx={ctx:>3}, head={str(arch):<10} -> val RMSE {rmse_v:6.2f} "
              f"({len(h.history['loss'])} ep, {t_train:.0f}s)")
    return results

print("=== Sweep — piecewise125 label ===")
sweep_pw = run_head_sweep("RUL_piecewise125")
best_pw = min(sweep_pw, key=lambda r: r["val_rmse"])
print(f"\nBest (piecewise): ctx={best_pw['ctx']}, head={best_pw['arch']}\n")

print("=== Sweep — linear label ===")
sweep_linear = run_head_sweep("RUL_linear")
best_linear = min(sweep_linear, key=lambda r: r["val_rmse"])
print(f"\nBest (linear): ctx={best_linear['ctx']}, head={best_linear['arch']}\n")

# --- 3d. Final training on ALL engines with best configs ---
def final_train_and_eval(best_cfg, label_col, label_name):
    ctx, arch = best_cfg["ctx"], best_cfg["arch"]
    y_max = float(train[label_col].max())
    E_trF, E_te = emb_cache[ctx]["E_trF"], emb_cache[ctx]["E_te"]
    y_trF = get_labels(train, label_col)
    keras.utils.set_random_seed(SEED)
    head = build_head(E_trF.shape[1], arch)
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                       restore_best_weights=True)
    t0 = time.time()
    head.fit(E_trF, y_trF / y_max, validation_split=0.1,
             epochs=EPOCHS, batch_size=BATCH, callbacks=[es], verbose=0)
    t_head = time.time() - t0
    y_pred = head.predict(E_te, verbose=0).ravel() * y_max
    res = evaluate_model(y_true, y_pred,
                         model_name=f"Chronos-2 frozen + head (ctx={ctx}, {arch})",
                         label_name=label_name)
    res["head_train_sec"] = round(t_head, 2)
    res["extract_min_ctx"] = round(extract_times[ctx] / 60, 1)
    res["config"] = f"ctx={ctx},head={arch}"
    return res, y_pred

result_piecewise, y_pred_piecewise = final_train_and_eval(
    best_pw, "RUL_piecewise125", "piecewise125")
result_linear, y_pred_linear = final_train_and_eval(
    best_linear, "RUL_linear", "linear")

# --- 3e. Full ladder ---
print("=" * 76)
print("FULL LADDER — piecewise125 label")
print("=" * 76)
print(f"{'':<16} {'LR':>10} {'XGBoost':>10} {'LSTM':>10} {'Chronos':>12}")
print("-" * 76)
print(f"{'MAE':<16} {17.79:>10.2f} {13.21:>10.2f} {10.69:>10.2f} {result_piecewise['mae']:>12.2f}")
print(f"{'RMSE':<16} {22.07:>10.2f} {17.71:>10.2f} {14.78:>10.2f} {result_piecewise['rmse']:>12.2f}")
print(f"{'NASA Score':<16} {1334.57:>10.2f} {784.53:>10.2f} {402.11:>10.2f} {result_piecewise['nasa_score']:>12.2f}")
print(f"{'Cost':<16} {'0.02s CPU':>10} {'1.6s CPU':>10} {'81s GPU':>10} "
      f"{str(result_piecewise['extract_min_ctx'])+'min+'+str(result_piecewise['head_train_sec'])+'s':>12}")
print("=" * 76)
print(f"\nChronos linear label: MAE {result_linear['mae']}, "
      f"RMSE {result_linear['rmse']}, NASA {result_linear['nasa_score']}")

In [ ]:
# ============================================================
# 06_Chronos.ipynb
# Section 4 — Visualization, Zone Analysis, Bias Test
# Purpose: Same diagnostics as NB04/05; answer why Chronos wins
#          the NASA Score but not MAE (near-failure bias analysis).
# ============================================================

# --- 4a. Predicted vs True scatter — both labels ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, y_pred, label_name, res in [
    (axes[0], y_pred_linear,    "Linear RUL label",        result_linear),
    (axes[1], y_pred_piecewise, "Piecewise-125 RUL label", result_piecewise),
]:
    ax.scatter(y_true, y_pred, alpha=0.6, s=35, color="#2E75B6",
               edgecolor="white", linewidth=0.5)
    lims = [0, max(y_true.max(), y_pred.max()) * 1.05]
    ax.plot(lims, lims, "r--", linewidth=1.2, label="Perfect prediction")
    ax.set_xlabel("True RUL (cycles)")
    ax.set_ylabel("Predicted RUL (cycles)")
    ax.set_title(f"Chronos-2 frozen + head — {label_name}\nMAE {res['mae']:.2f} | "
                 f"RMSE {res['rmse']:.2f} | NASA {res['nasa_score']:.0f}")
    ax.legend(loc="upper left")
    ax.set_xlim(lims); ax.set_ylim(lims)

fig.suptitle("FD001 — Chronos-2: predicted vs. true RUL (100 test engines, last cycle)",
             fontsize=13, y=1.02)
fig.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "fd001_chronos_pred_vs_true.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}\n")

# --- 4b. Error vs True RUL ---
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
for ax, y_pred, label_name in [
    (axes2[0], y_pred_linear,    "Linear RUL label"),
    (axes2[1], y_pred_piecewise, "Piecewise-125 RUL label"),
]:
    errors = y_pred - y_true
    ax.scatter(y_true, errors, alpha=0.6, s=35, color="#D9822B",
               edgecolor="white", linewidth=0.5)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_xlabel("True RUL (cycles)")
    ax.set_ylabel("Prediction error (pred − true, cycles)")
    ax.set_title(f"{label_name}\nAbove 0 = late (dangerous), below 0 = early (conservative)")

fig2.suptitle("FD001 — Chronos-2: prediction error vs. true RUL", fontsize=13, y=1.03)
fig2.tight_layout()
fig2_path = os.path.join(FIGURES_DIR, "fd001_chronos_error_vs_true.png")
fig2.savefig(fig2_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig2_path}\n")

# --- 4c. Zone analysis — all four models (piecewise) ---
print("=== Zone analysis — piecewise label, all four models ===\n")
errors_pw = y_pred_piecewise - y_true

BASELINES = {
    "Near failure (RUL <= 30)":  {"LR": +18.77, "XGB": +8.59, "LSTM": +0.24},
    "Mid life (30 < RUL <= 80)": {"LR": +19.75, "XGB": +8.83, "LSTM": +5.58},
    "Early life (RUL > 80)":     {"LR": -8.46,  "XGB": -3.21, "LSTM": -2.34},
}
zones = [("Near failure (RUL <= 30)",  y_true <= 30),
         ("Mid life (30 < RUL <= 80)", (y_true > 30) & (y_true <= 80)),
         ("Early life (RUL > 80)",     y_true > 80)]

print(f"{'Zone':<28} {'n':>4} {'Chr MAE':>8} {'Chr bias':>9} "
      f"{'LSTM':>7} {'XGB':>7} {'LR':>7}")
print("-" * 76)
for zone_name, mask in zones:
    n = mask.sum()
    z_mae  = np.abs(errors_pw[mask]).mean()
    z_bias = errors_pw[mask].mean()
    b = BASELINES[zone_name]
    print(f"{zone_name:<28} {n:>4} {z_mae:>8.2f} {z_bias:>+9.2f} "
          f"{b['LSTM']:>+7.2f} {b['XGB']:>+7.2f} {b['LR']:>+7.2f}")

# --- 4d. Top 5 NASA contributors — linear label ---
print("\n=== Top 5 NASA-Score contributors — linear label ===\n")
d = y_pred_linear - y_true
scores = np.where(d < 0, np.exp(-d / 13.0) - 1.0, np.exp(d / 10.0) - 1.0)
top5 = np.argsort(scores)[::-1][:5]
print(f"{'unit':>5} {'true RUL':>9} {'predicted':>10} {'error':>8} {'score contrib':>14}")
print("-" * 52)
for i in top5:
    print(f"{test_last['unit'].values[i]:>5} {y_true[i]:>9.0f} "
          f"{y_pred_linear[i]:>10.2f} {d[i]:>+8.2f} {scores[i]:>14.1f}")
print(f"\nTop-5 share of total NASA Score: "
      f"{100 * scores[top5].sum() / scores.sum():.1f}%")

# --- 4e. Late-prediction comparison (the NASA-vs-MAE puzzle) ---
print("\n=== Why Chronos wins NASA: late-prediction analysis (piecewise) ===\n")
late_mask = errors_pw > 0
print(f"Chronos late predictions (pred > true): {late_mask.sum()}/100 engines")
print(f"  Mean late error (over-prediction): {errors_pw[late_mask].mean():+.2f} cycles")
print(f"  Max late error:                    {errors_pw[late_mask].max():+.2f} cycles")
print(f"  Near-failure (RUL<=30) late count: "
      f"{((y_true <= 30) & late_mask).sum()}/{(y_true <= 30).sum()}")

In [ ]:
# --- 4f. Verification: per-engine NASA contributions, LSTM vs Chronos ---
lstm_preds = pd.read_csv(os.path.join(RESULTS_DIR, "fd001_lstm_predictions.csv"))
d_lstm = lstm_preds["pred_LSTM_piecewise125"].values - lstm_preds["RUL_true"].values
d_chr  = y_pred_piecewise - y_true

for name, d in [("LSTM", d_lstm), ("Chronos", d_chr)]:
    s = np.where(d < 0, np.exp(-d/13.0) - 1.0, np.exp(d/10.0) - 1.0)
    top3 = np.argsort(s)[::-1][:3]
    print(f"{name}: total NASA {s.sum():.2f} | max late error {d.max():+.2f} | "
          f"top-3 contributions: {np.round(s[top3], 1).tolist()} "
          f"(= {100*s[top3].sum()/s.sum():.0f}% of total)")

In [ ]:
# ============================================================
# 06_Chronos.ipynb
# Section 5 — Save Predictions, Sweep Table, and Finalize
# Purpose: Save per-engine predictions + tuning sweep to results/,
#          print final summary. Closes the modeling phase.
# ============================================================

# --- 5a. Per-engine predictions table ---
predictions = pd.DataFrame({
    "unit": test_last["unit"].values,
    "cycle_last": test_last["cycle"].values,
    "RUL_true": y_true,
    "pred_Chronos_linear": np.round(y_pred_linear, 2),
    "pred_Chronos_piecewise125": np.round(y_pred_piecewise, 2),
})
predictions["error_linear"] = np.round(
    predictions["pred_Chronos_linear"] - predictions["RUL_true"], 2)
predictions["error_piecewise125"] = np.round(
    predictions["pred_Chronos_piecewise125"] - predictions["RUL_true"], 2)

print("Per-engine predictions (first 10 rows):")
print(predictions.head(10).to_string(index=False))

pred_path = os.path.join(RESULTS_DIR, "fd001_chronos_predictions.csv")
predictions.to_csv(pred_path, index=False)
print(f"\nSaved predictions to:\n{pred_path}")

# --- 5b. Save the tuning sweep table (both labels) ---
sweep_rows = []
for r in sweep_pw:
    sweep_rows.append({"context": r["ctx"], "head": str(r["arch"]),
                       "label": "piecewise125",
                       "val_rmse": round(r["val_rmse"], 2),
                       "epochs_run": r["epochs_run"],
                       "head_train_s": round(r["head_train_s"], 1)})
for r in sweep_linear:
    sweep_rows.append({"context": r["ctx"], "head": str(r["arch"]),
                       "label": "linear",
                       "val_rmse": round(r["val_rmse"], 2),
                       "epochs_run": r["epochs_run"],
                       "head_train_s": round(r["head_train_s"], 1)})
sweep_df = pd.DataFrame(sweep_rows)

sweep_df["selected"] = False
for label, best in [("piecewise125", best_pw), ("linear", best_linear)]:
    mask = ((sweep_df["label"] == label) &
            (sweep_df["context"] == best["ctx"]) &
            (sweep_df["head"] == str(best["arch"])))
    sweep_df.loc[mask, "selected"] = True

sweep_path = os.path.join(RESULTS_DIR, "fd001_chronos_tuning_sweep.csv")
sweep_df.to_csv(sweep_path, index=False)
print(f"\nSaved tuning sweep to:\n{sweep_path}\n")
print("Tuning sweep table:")
print(sweep_df.to_string(index=False))

# --- 5c. Extraction-time record (cost accounting, Decision 5) ---
extract_df = pd.DataFrame([
    {"context": ctx, "extract_time_min": round(t / 60, 1)}
    for ctx, t in extract_times.items()
])
extract_path = os.path.join(RESULTS_DIR, "fd001_chronos_extraction_times.csv")
extract_df.to_csv(extract_path, index=False)
print(f"\nSaved extraction times to:\n{extract_path}")
print(extract_df.to_string(index=False))

# --- 5d. Final summary block ---
print("\n" + "=" * 60)
print("NOTEBOOK 06 COMPLETE — MODELING PHASE FINISHED")
print("=" * 60)
print(f"""
Model: Chronos-2 (frozen, amazon/chronos-2, 478 MB) + regression head
Embeddings: native multivariate .embed(), mean-pooled to 768 dims
Best config (piecewise): ctx=100, head (128, 64)
Best config (linear):    ctx=100, head (64,)
Head training: label-scaled targets, Adam lr=0.001, batch 64,
early stopping patience 10, seed 42.
Test protocol: last available cycle per test engine (n=100)

Run 1 — linear label:
  MAE {result_linear['mae']}, RMSE {result_linear['rmse']}, NASA {result_linear['nasa_score']}
Run 2 — piecewise125 label:
  MAE {result_piecewise['mae']}, RMSE {result_piecewise['rmse']}, NASA {result_piecewise['nasa_score']}

Cost: 4.1 min embedding extraction per context (GPU T4)
      + {result_piecewise['head_train_sec']}s head training (piecewise)

Files saved:
  results/fd001_chronos_predictions.csv
  results/fd001_chronos_tuning_sweep.csv
  results/fd001_chronos_extraction_times.csv
  figures/fd001_chronos_pred_vs_true.png
  figures/fd001_chronos_error_vs_true.png
""")
print("=" * 60)
print("ALL FOUR MODELS COMPLETE: LR -> XGBoost -> LSTM -> Chronos-2")
print("Experiment log: EXP-001 through EXP-008")
print("=" * 60)

In [ ]:
# Inspect exactly what pipe2.embed returns for a small batch,
# so the pooling logic can be written against the real structure.
import numpy as np, torch

probe = [tr_seqs[i] for i in range(4)]
L = max(len(s) for s in probe)
arr = np.stack([
    np.vstack([np.repeat(s[:1], L - len(s), axis=0), s]) if len(s) < L else s
    for s in probe
])
x = torch.tensor(arr, dtype=torch.float32).transpose(1, 2)
print(f"input to embed: {tuple(x.shape)}  (batch, n_sensors, length)\n")

out = pipe2.embed(x)
print(f"type: {type(out)}")

def describe(o, name="out", depth=0):
    pad = "  " * depth
    if isinstance(o, (list, tuple)):
        print(f"{pad}{name}: {type(o).__name__} of {len(o)}")
        for i, item in enumerate(o[:3]):
            describe(item, f"[{i}]", depth + 1)
    elif hasattr(o, "shape"):
        print(f"{pad}{name}: {type(o).__name__} shape {tuple(o.shape)} dtype {o.dtype}")
    else:
        print(f"{pad}{name}: {type(o).__name__} -> {str(o)[:80]}")

describe(out)

In [ ]:
# ============================================================
# NB06 — Section 6: FAIR comparison — Chronos-2 on FULL history
# Same pipeline, just window = MAX_LEN instead of 50.
# pre-padding (already in build_windows_from_df) unifies lengths.
# ============================================================
import os
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

RESULTS_DIR = os.path.join(DATA_DIR, "..", "results")
Y_MAX_LABEL = 125.0

# Longest engine in TRAIN = full-history window length
MAX_LEN = int(train.groupby("unit").size().max())
print(f"Full-history window length (MAX_LEN) = {MAX_LEN}")

# --- Build FULL-history windows (train) using the SAME builder ---
print("\nBuilding full-history TRAIN windows...")
X_tr_win = build_windows_from_df(train, window=MAX_LEN)
y_tr = (train.sort_values(["unit","cycle"])
              .groupby("unit", group_keys=False)
              .apply(lambda g: g)["RUL_piecewise125"].values.astype(np.float32))
print(f"  X_tr_win: {X_tr_win.shape}  (n, {MAX_LEN}, {N_FEATURES})")
print(f"  y_tr:     {y_tr.shape}")

# --- Build FULL-history windows (test): last observed cycle per engine ---
# For test we want ONE window per engine = its full observed history.
def build_test_full_windows(df, units_order, window):
    X_list = []
    for unit in units_order:
        g = df[df["unit"]==unit].sort_values("cycle")
        feats = g[FINAL_FEATURES_FD001].values
        seq = feats[-window:] if len(feats) >= window else feats
        if len(seq) < window:
            seq = np.vstack([np.zeros((window-len(seq), N_FEATURES)), seq])
        X_list.append(seq)
    return np.array(X_list, dtype=np.float32)

units_test = test_last["unit"].values
y_true = test_last["RUL_true"].values
X_te_win = build_test_full_windows(test_full, units_test, MAX_LEN)
print(f"  X_te_win: {X_te_win.shape}")

# --- Extract embeddings with the SAME function (batched) ---
print("\nExtracting embeddings (full history)...")
Xtr_emb = extract_embeddings(X_tr_win, batch_size=64, desc="train-full")
Xte_emb = extract_embeddings(X_te_win, batch_size=64, desc="test-full")
print(f"  Xtr_emb: {Xtr_emb.shape} | Xte_emb: {Xte_emb.shape}")

# --- Train head (Ridge) + evaluate ---
scaler = StandardScaler().fit(Xtr_emb)
head = Ridge(alpha=10.0).fit(scaler.transform(Xtr_emb), y_tr)
y_pred = np.clip(head.predict(scaler.transform(Xte_emb)), 0, 125)

res = evaluate_model(y_true, y_pred, "Chronos-2 (full history)", "piecewise125")

# --- Save + fair table ---
pd.DataFrame({"unit": units_test, "RUL_true": y_true,
              "pred_Chronos_full": y_pred}).to_csv(
    os.path.join(RESULTS_DIR, "fd001_chronos_full_history.csv"), index=False)

print("="*70)
print("FAIR COMPARISON — every model sees each engine's FULL history")
print("="*70)
print(f"{'Model':<18}{'Input':<20}{'MAE':>7}{'RMSE':>7}{'NASA':>9}")
print("-"*70)
print(f"{'LR':<18}{'65 features':<20}{14.99:>7.2f}{17.98:>7.2f}{514.27:>9.2f}")
print(f"{'XGBoost':<18}{'65 features':<20}{8.57:>7.2f}{12.01:>7.2f}{242.67:>9.2f}")
print(f"{'LSTM (full)':<18}{'all cycles':<20}{10.16:>7.2f}{13.79:>7.2f}{313.54:>9.2f}")
print(f"{'Chronos (full)':<18}{'all cycles':<20}{res['mae']:>7.2f}{res['rmse']:>7.2f}{res['nasa_score']:>9.2f}")
print("-"*70)

In [ ]:
# ============================================================
# NB10B — Chronos-2 on the full observed history, without global padding
# Change from the earlier version: sequences are no longer padded to a
# common length of 362. Because the sensor values are scaled to [0, 1],
# a zero is itself a plausible reading, so padded positions could not be
# distinguished from genuine ones. Here each window is passed at its own
# observed length, and windows are grouped into length buckets so that
# any residual padding within a batch spans a few cycles rather than
# hundreds.
# Backbone is frozen throughout; only the Ridge head is fitted.
# ============================================================
import os, time
import numpy as np, pandas as pd, torch
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

RESULTS_DIR = os.path.join(DATA_DIR, "..", "results")
EMB_DIR     = "{DATA_DIR}/../embeddings_nopad"
os.makedirs(EMB_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

BATCH_TARGET = 32          # windows per batch, reduced for long sequences
MIN_LEN      = 2           # a single cycle carries no temporal information

def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0)-1.0, np.exp(d/10.0)-1.0)))

# --- Collect variable-length sequences, one per engine-cycle ---
def collect_sequences(df, per_engine_last_only=False, units_order=None):
    """Returns a list of (n_cycles, n_sensors) arrays and their labels."""
    seqs, labels, units = [], [], []
    order = units_order if units_order is not None else sorted(df["unit"].unique())
    for unit in order:
        g = df[df["unit"] == unit].sort_values("cycle")
        f = g[FINAL_FEATURES_FD001].values.astype(np.float32)
        if per_engine_last_only:
            seqs.append(f); units.append(unit)
        else:
            lab = g["RUL_piecewise125"].values.astype(np.float32)
            for end in range(MIN_LEN - 1, len(g)):
                seqs.append(f[:end + 1]); labels.append(lab[end]); units.append(unit)
    return seqs, np.array(labels, np.float32), np.array(units)

print("Collecting variable-length sequences...")
tr_seqs, y_tr, tr_units = collect_sequences(train)
units_test = test_last["unit"].values
y_true = test_last["RUL_true"].values
te_seqs, _, _ = collect_sequences(test_full, per_engine_last_only=True,
                                  units_order=units_test)
tr_lens = np.array([len(s) for s in tr_seqs])
print(f"  train: {len(tr_seqs)} windows | lengths {tr_lens.min()}–{tr_lens.max()}")
print(f"  test:  {len(te_seqs)} windows | lengths "
      f"{min(len(s) for s in te_seqs)}–{max(len(s) for s in te_seqs)}")

# --- Length-bucketed embedding extraction ---
def embed_bucketed(seqs, desc, batch_target=BATCH_TARGET):
    """Sort by length, batch neighbours together, and pad only within a
    batch. Padding repeats the first observed value rather than inserting
    zeros, so a padded position resembles a continuation of the record.
    Results are returned in the original order."""
    order = np.argsort([len(s) for s in seqs], kind="stable")
    out = [None] * len(seqs)
    t0, done = time.time(), 0
    for start in range(0, len(order), batch_target):
        idx = order[start:start + batch_target]
        batch = [seqs[i] for i in idx]
        L = max(len(s) for s in batch)
        arr = np.stack([
            np.vstack([np.repeat(s[:1], L - len(s), axis=0), s]) if len(s) < L else s
            for s in batch
        ])                                                  # (b, L, n_sensors)
        x = torch.tensor(arr, dtype=torch.float32).transpose(1, 2)   # (b, n, L)
        with torch.no_grad():
            reps, _ = pipe2.embed(x)
        # `reps` is a list with one entry per window. Each entry has shape
        # (n_sensors, n_patches, 768): the model represents every sensor
        # separately and splits it into patches, whose number depends on
        # the window length. Averaging over both axes yields one
        # fixed-length vector per window, so windows of different lengths
        # become comparable.
        for k, i in enumerate(idx):
            out[i] = reps[k].mean(dim=(0, 1)).float().cpu().numpy()
        done += len(idx)
        if start % (batch_target * 40) == 0:
            el = time.time() - t0
            print(f"  {desc}: {done}/{len(seqs)} | {el:.0f}s elapsed | "
                  f"~{el / max(done,1) * (len(seqs) - done):.0f}s left")
    return np.stack(out)

emb_train_path = os.path.join(EMB_DIR, "emb_train_nopad.npy")
emb_test_path  = os.path.join(EMB_DIR, "emb_test_nopad.npy")

if os.path.exists(emb_train_path) and os.path.exists(emb_test_path):
    Xtr_emb = np.load(emb_train_path); Xte_emb = np.load(emb_test_path)
    print(f"\nLoaded cached embeddings: {Xtr_emb.shape}, {Xte_emb.shape}")
else:
    print("\nExtracting embeddings (train)...")
    t0 = time.time()
    Xtr_emb = embed_bucketed(tr_seqs, "train")
    t_train = time.time() - t0
    print("Extracting embeddings (test)...")
    Xte_emb = embed_bucketed(te_seqs, "test")
    np.save(emb_train_path, Xtr_emb); np.save(emb_test_path, Xte_emb)
    dev = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    print(f"\nExtraction time (train): {t_train:.0f}s on {dev}")
    print(f"Saved to {EMB_DIR}")

print(f"Xtr_emb {Xtr_emb.shape} | Xte_emb {Xte_emb.shape}")

# --- Fit the head. Scaler and Ridge see training embeddings only. ---
scaler = StandardScaler().fit(Xtr_emb)
head = Ridge(alpha=10.0).fit(scaler.transform(Xtr_emb), y_tr)
y_pred = np.clip(head.predict(scaler.transform(Xte_emb)), 0, 125)

print(f"\nChronos-2 (full history, no global padding): "
      f"MAE {mean_absolute_error(y_true, y_pred):.2f} | "
      f"RMSE {np.sqrt(mean_squared_error(y_true, y_pred)):.2f} | "
      f"NASA {nasa_score(y_true, y_pred):.2f}")

pd.DataFrame({"unit": units_test, "RUL_true": y_true,
              "pred_Chronos_full": y_pred}).to_csv(
    os.path.join(RESULTS_DIR, "fd001_chronos_full_history.csv"), index=False)
print("Saved: fd001_chronos_full_history.csv")

In [ ]:
# ============================================================
# Final Fair Comparison — official figure
# بنثبت: الموديل البسيط بالـ features غلب المعقّدين على نفس المعلومات
# العدالة: كل الموديلات شافت التاريخ الكامل (مكتوب على الرسمة)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import os

FIGURES_DIR = os.path.join(DATA_DIR, "..", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

# --- The fair results (all models see FULL history) ---
models = ["Linear\nRegression", "XGBoost", "LSTM", "Chronos-2"]
mae    = [14.99, 8.57, 10.16, 9.34]
nasa   = [514.27, 242.67, 313.54, 244.60]
# input type: features (tabular) vs full sequence (temporal)
input_type = ["features", "features", "sequence", "sequence"]
colors = ["#999999", "#2E75B6", "#C55A11", "#548235"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ---- Left: MAE ----
bars1 = ax1.bar(models, mae, color=colors, edgecolor="black", linewidth=1.2)
for b, v, t in zip(bars1, mae, input_type):
    ax1.annotate(f"{v:.2f}", (b.get_x()+b.get_width()/2, v),
                 textcoords="offset points", xytext=(0, 5),
                 ha="center", fontweight="bold", fontsize=11)
    ax1.annotate(f"({t})", (b.get_x()+b.get_width()/2, 0.5),
                 textcoords="offset points", xytext=(0, 0),
                 ha="center", fontsize=8, color="white", fontweight="bold")
ax1.set_ylabel("MAE (cycles) — lower is better", fontsize=11)
ax1.set_title("Mean Absolute Error", fontsize=12, fontweight="bold")
ax1.grid(True, axis="y", alpha=0.3)
# highlight the winner
bars1[1].set_edgecolor("#C00000"); bars1[1].set_linewidth(3)

# ---- Right: NASA Score ----
bars2 = ax2.bar(models, nasa, color=colors, edgecolor="black", linewidth=1.2)
for b, v in zip(bars2, nasa):
    ax2.annotate(f"{v:.1f}", (b.get_x()+b.get_width()/2, v),
                 textcoords="offset points", xytext=(0, 5),
                 ha="center", fontweight="bold", fontsize=11)
ax2.set_ylabel("NASA Score — lower is better", fontsize=11)
ax2.set_title("NASA Asymmetric Score", fontsize=12, fontweight="bold")
ax2.grid(True, axis="y", alpha=0.3)
bars2[1].set_edgecolor("#C00000"); bars2[1].set_linewidth(3)

fig.suptitle("Fair Comparison — every model sees each engine's FULL history\n"
             "XGBoost (features) outperforms the complex sequence models",
             fontsize=13, fontweight="bold")
plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "fd001_fair_comparison_final.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

In [ ]:
# ============================================================
# 06 — Save/Load full-history embeddings (survive session drops)
# ============================================================
import os, numpy as np
EMB_DIR = os.path.join(DATA_DIR, "..", "embeddings")
os.makedirs(EMB_DIR, exist_ok=True)

tr_path = os.path.join(EMB_DIR, "chronos_full_train_emb.npy")
te_path = os.path.join(EMB_DIR, "chronos_full_test_emb.npy")
ytr_path = os.path.join(EMB_DIR, "chronos_full_y_tr.npy")

# If embeddings are in memory -> SAVE them. If not -> LOAD from Drive.
if "Xtr_emb" in dir() and "Xte_emb" in dir():
    np.save(tr_path, Xtr_emb)
    np.save(te_path, Xte_emb)
    np.save(ytr_path, y_tr)
    print(f"✓ SAVED embeddings to Drive:")
    print(f"    train {Xtr_emb.shape}, test {Xte_emb.shape}")
else:
    Xtr_emb = np.load(tr_path)
    Xte_emb = np.load(te_path)
    y_tr    = np.load(ytr_path)
    print(f"✓ LOADED embeddings from Drive:")
    print(f"    train {Xtr_emb.shape}, test {Xte_emb.shape}")

In [ ]:
import os, numpy as np
for d in ["embeddings_full", "embeddings", "embeddings_nopad"]:
    p = f"{DATA_DIR}/../{d}"
    if os.path.exists(p):
        print(f"\n{d}/")
        for f in sorted(os.listdir(p)):
            fp = os.path.join(p, f)
            if f.endswith(".npy"):
                print(f"  {f}  shape {np.load(fp, mmap_mode='r').shape}")
            else:
                print(f"  {f}")
    else:
        print(f"\n{d}/ — not found")

In [ ]:
# Load the cached full-history embeddings produced with uniform padding.
# This replaces re-running the extraction, which takes about ten minutes.
import os
import numpy as np, pandas as pd

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
EMB_DIR       = os.path.join(PROJECT_ROOT, "embeddings")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

Xtr_emb = np.load(os.path.join(EMB_DIR, "chronos_full_train_emb.npy"))
Xte_emb = np.load(os.path.join(EMB_DIR, "chronos_full_test_emb.npy"))
y_tr    = np.load(os.path.join(EMB_DIR, "chronos_full_y_tr.npy"))

train     = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))
units_test = test_last["unit"].values
y_true = test_last["RUL_true"].values

def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0)-1.0, np.exp(d/10.0)-1.0)))

print(f"Xtr_emb {Xtr_emb.shape} | Xte_emb {Xte_emb.shape} | y_tr {y_tr.shape}")
print(f"train {train.shape} | {len(units_test)} test engines")
assert len(Xtr_emb) == len(train), "embedding rows do not match the training rows"
print("Row counts match — ready for the multi-seed cell.")

In [ ]:
# ============================================================
# NB06 — Multi-seed Chronos-2 head, saving per-seed predictions
# Change from the earlier version: the prediction of every seed is saved.
# Everything else is unchanged: the engine-level early-stopping split,
# the head architecture, and the training settings.
# Embeddings are loaded from cache; the frozen backbone is not re-run.
# ============================================================
import os, sys, time
import numpy as np, pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
EMB_DIR       = os.path.join(PROJECT_ROOT, "embeddings")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
PRED_DIR      = os.path.join(RESULTS_DIR, "per_seed")
os.makedirs(PRED_DIR, exist_ok=True)

SEEDS = [42, 43, 44, 45, 46]
N_INNER, SPLIT_SEED, Y_MAX = 10, 7, 125.0

def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0)-1.0, np.exp(d/10.0)-1.0)))

Xtr_emb = np.load(os.path.join(EMB_DIR, "chronos_full_train_emb.npy"))
Xte_emb = np.load(os.path.join(EMB_DIR, "chronos_full_test_emb.npy"))
y_tr    = np.load(os.path.join(EMB_DIR, "chronos_full_y_tr.npy"))

train     = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))
units_test = test_last["unit"].values
y_true = test_last["RUL_true"].values
print(f"Xtr_emb {Xtr_emb.shape} | Xte_emb {Xte_emb.shape}")

# Engine id for every training row, in the same order as the embeddings
row_units = np.concatenate([np.full(len(g), unit) for unit, g in train.groupby("unit")])
assert len(row_units) == len(Xtr_emb), \
    f"order mismatch: {len(row_units)} engine ids for {len(Xtr_emb)} rows"

rng = np.random.default_rng(SPLIT_SEED)
inner_units = rng.choice(np.sort(train["unit"].unique()), size=N_INNER, replace=False)
is_val = np.isin(row_units, inner_units)
print(f"Early-stopping split: {100 - N_INNER} engines ({(~is_val).sum()} rows) fit | "
      f"{N_INNER} engines ({is_val.sum()} rows) validate")
print(f"Held-out engines: {sorted(int(u) for u in inner_units)}\n")

# Standardisation is estimated from the fitting portion only
scaler = StandardScaler().fit(Xtr_emb[~is_val])
Xfit_s = scaler.transform(Xtr_emb[~is_val])
Xval_s = scaler.transform(Xtr_emb[is_val])
Xte_s  = scaler.transform(Xte_emb)
yfit, yval = y_tr[~is_val], y_tr[is_val]

def build_head_seed(input_dim, arch=(128, 64)):
    L = [layers.Input(shape=(input_dim,))]
    for n in arch:
        L += [layers.Dense(n, activation="relu"), layers.Dropout(0.2)]
    L += [layers.Dense(1)]
    m = keras.Sequential(L)
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="mse")
    return m

runs, preds = [], {}
for seed in SEEDS:
    keras.utils.set_random_seed(seed)
    head = build_head_seed(Xfit_s.shape[1])
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                       restore_best_weights=True)
    t0 = time.time()
    head.fit(Xfit_s, yfit / Y_MAX,
             validation_data=(Xval_s, yval / Y_MAX),
             epochs=100, batch_size=64, callbacks=[es], verbose=0)
    t_train = time.time() - t0

    p = np.clip(head.predict(Xte_s, verbose=0).ravel() * Y_MAX, 0, Y_MAX)
    preds[seed] = p
    runs.append({"seed": seed,
                 "mae": mean_absolute_error(y_true, p),
                 "rmse": np.sqrt(mean_squared_error(y_true, p)),
                 "nasa": nasa_score(y_true, p),
                 "train_s": t_train})
    print(f"  seed {seed}: MAE {runs[-1]['mae']:6.2f} | RMSE {runs[-1]['rmse']:6.2f} | "
          f"NASA {runs[-1]['nasa']:8.2f} | {t_train:.0f}s")

df = pd.DataFrame(runs)
print(f"\nChronos-2 (full history) across seeds {SEEDS}:")
for m_ in ["mae", "rmse", "nasa"]:
    print(f"  {m_.upper():>5}: {df[m_].mean():8.2f} ± {df[m_].std():5.2f} "
          f"(min {df[m_].min():.2f}, max {df[m_].max():.2f})")
print(f"  Mean head training time: {df['train_s'].mean():.0f}s")

df["model"] = "Chronos_full"
df.to_csv(os.path.join(RESULTS_DIR, "fd001_multiseed_chronos_full.csv"), index=False)

wide = pd.DataFrame({"unit": units_test, "RUL_true": y_true})
for seed in SEEDS:
    wide[f"pred_seed{seed}"] = preds[seed]
wide.to_csv(os.path.join(PRED_DIR, "chronos_per_seed.csv"), index=False)

pd.DataFrame({"unit": units_test, "RUL_true": y_true,
              "pred_Chronos_full": preds[SEEDS[0]]}).to_csv(
    os.path.join(RESULTS_DIR, "fd001_chronos_full_history.csv"), index=False)

print("\nSaved: fd001_multiseed_chronos_full.csv + per_seed/chronos_per_seed.csv")

**# ============================================================
# 06 — Multi-seed robustness: Chronos-2 head (FULL history)
# Change from the earlier version: the early-stopping split is now
# defined over engines rather than over rows. Keras' validation_split
# takes the final fraction of the ordered array, and because the
# embeddings are ordered by engine that cut can fall inside one engine's
# trajectory, so the same engine would appear in both parts. Here ten
# engines are set aside in advance and never contribute a training row.
# Everything else is unchanged: architecture, dropout, epochs, batch
# size, learning rate, patience, and seeds.
# ============================================================
import os, time
import numpy as np, pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

RESULTS_DIR = os.path.join(DATA_DIR, "..", "results")
SEEDS = [42, 43, 44, 45, 46]
Y_MAX = 125.0
N_INNER, SPLIT_SEED = 10, 7      # engines held out for early stopping

# needs from memory: Xtr_emb, Xte_emb, y_tr, y_true, nasa_score, train

# --- Engine id for every training row, in the same order as Xtr_emb ---
row_units = np.concatenate([
    np.full(len(g), unit) for unit, g in train.groupby("unit")
])
assert len(row_units) == len(Xtr_emb), \
    f"order mismatch: {len(row_units)} engine ids for {len(Xtr_emb)} rows"

rng = np.random.default_rng(SPLIT_SEED)
inner_units = rng.choice(np.sort(train["unit"].unique()), size=N_INNER, replace=False)
is_val = np.isin(row_units, inner_units)
print(f"Early-stopping split: {100 - N_INNER} engines ({(~is_val).sum()} rows) fit | "
      f"{N_INNER} engines ({is_val.sum()} rows) validate")
print(f"Held-out engines: {sorted(int(u) for u in inner_units)}\n")

# Scale on the fitting portion only, so the held-out engines do not
# influence the standardisation either.
scaler = StandardScaler().fit(Xtr_emb[~is_val])
Xfit_s = scaler.transform(Xtr_emb[~is_val])
Xval_s = scaler.transform(Xtr_emb[is_val])
Xte_s  = scaler.transform(Xte_emb)
yfit, yval = y_tr[~is_val], y_tr[is_val]

def build_head_seed(input_dim, arch=(128, 64)):
    L = [layers.Input(shape=(input_dim,))]
    for n in arch:
        L += [layers.Dense(n, activation="relu"), layers.Dropout(0.2)]
    L += [layers.Dense(1)]
    m = keras.Sequential(L)
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="mse")
    return m

chronos_full_runs = []
for seed in SEEDS:
    keras.utils.set_random_seed(seed)
    head = build_head_seed(Xfit_s.shape[1])
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                       restore_best_weights=True)
    t0 = time.time()
    head.fit(Xfit_s, yfit / Y_MAX,
             validation_data=(Xval_s, yval / Y_MAX),
             epochs=100, batch_size=64, callbacks=[es], verbose=0)
    t_train = time.time() - t0
    y_pred = np.clip(head.predict(Xte_s, verbose=0).ravel() * Y_MAX, 0, 125)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    ns   = nasa_score(y_true, y_pred)
    chronos_full_runs.append({"seed": seed, "mae": mae, "rmse": rmse,
                              "nasa": ns, "train_s": t_train})
    print(f"  Chronos(full) seed {seed}: MAE {mae:6.2f} | RMSE {rmse:6.2f} | "
          f"NASA {ns:8.2f} | {t_train:.0f}s")
    if seed == SEEDS[0]:
        pd.DataFrame({"unit": units_test, "RUL_true": y_true,
                      "pred_Chronos_full": y_pred}).to_csv(
            os.path.join(RESULTS_DIR, "fd001_chronos_full_history.csv"), index=False)

df_chr_full = pd.DataFrame(chronos_full_runs)
print(f"\nChronos-2 (FULL history) across seeds {SEEDS}:")
for m in ["mae", "rmse", "nasa"]:
    print(f"  {m.upper():>5}: {df_chr_full[m].mean():8.2f} ± {df_chr_full[m].std():5.2f} "
          f"(min {df_chr_full[m].min():.2f}, max {df_chr_full[m].max():.2f})")

df_chr_full["model"] = "Chronos_full"
df_chr_full.to_csv(os.path.join(RESULTS_DIR, "fd001_multiseed_chronos_full.csv"), index=False)
print("\nSaved: fd001_multiseed_chronos_full.csv + fd001_chronos_full_history.csv")**

In [ ]:
# ============================================================
# 06 — Multi-seed robustness: Chronos-2 head (FULL history)
# Change from the earlier version: the early-stopping split is now
# defined over engines rather than over rows. Keras' validation_split
# takes the final fraction of the ordered array, and because the
# embeddings are ordered by engine that cut can fall inside one engine's
# trajectory, so the same engine would appear in both parts. Here ten
# engines are set aside in advance and never contribute a training row.
# Everything else is unchanged: architecture, dropout, epochs, batch
# size, learning rate, patience, and seeds.
# ============================================================
import os, time
import numpy as np, pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

RESULTS_DIR = os.path.join(DATA_DIR, "..", "results")
SEEDS = [42, 43, 44, 45, 46]
Y_MAX = 125.0
N_INNER, SPLIT_SEED = 10, 7      # engines held out for early stopping

# needs from memory: Xtr_emb, Xte_emb, y_tr, y_true, nasa_score, train

# --- Engine id for every training row, in the same order as Xtr_emb ---
row_units = np.concatenate([
    np.full(len(g), unit) for unit, g in train.groupby("unit")
])
assert len(row_units) == len(Xtr_emb), \
    f"order mismatch: {len(row_units)} engine ids for {len(Xtr_emb)} rows"

rng = np.random.default_rng(SPLIT_SEED)
inner_units = rng.choice(np.sort(train["unit"].unique()), size=N_INNER, replace=False)
is_val = np.isin(row_units, inner_units)
print(f"Early-stopping split: {100 - N_INNER} engines ({(~is_val).sum()} rows) fit | "
      f"{N_INNER} engines ({is_val.sum()} rows) validate")
print(f"Held-out engines: {sorted(int(u) for u in inner_units)}\n")

# Scale on the fitting portion only, so the held-out engines do not
# influence the standardisation either.
scaler = StandardScaler().fit(Xtr_emb[~is_val])
Xfit_s = scaler.transform(Xtr_emb[~is_val])
Xval_s = scaler.transform(Xtr_emb[is_val])
Xte_s  = scaler.transform(Xte_emb)
yfit, yval = y_tr[~is_val], y_tr[is_val]

def build_head_seed(input_dim, arch=(128, 64)):
    L = [layers.Input(shape=(input_dim,))]
    for n in arch:
        L += [layers.Dense(n, activation="relu"), layers.Dropout(0.2)]
    L += [layers.Dense(1)]
    m = keras.Sequential(L)
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="mse")
    return m

chronos_full_runs = []
for seed in SEEDS:
    keras.utils.set_random_seed(seed)
    head = build_head_seed(Xfit_s.shape[1])
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                       restore_best_weights=True)
    t0 = time.time()
    head.fit(Xfit_s, yfit / Y_MAX,
             validation_data=(Xval_s, yval / Y_MAX),
             epochs=100, batch_size=64, callbacks=[es], verbose=0)
    t_train = time.time() - t0
    y_pred = np.clip(head.predict(Xte_s, verbose=0).ravel() * Y_MAX, 0, 125)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    ns   = nasa_score(y_true, y_pred)
    chronos_full_runs.append({"seed": seed, "mae": mae, "rmse": rmse,
                              "nasa": ns, "train_s": t_train})
    print(f"  Chronos(full) seed {seed}: MAE {mae:6.2f} | RMSE {rmse:6.2f} | "
          f"NASA {ns:8.2f} | {t_train:.0f}s")
    if seed == SEEDS[0]:
        pd.DataFrame({"unit": units_test, "RUL_true": y_true,
                      "pred_Chronos_full": y_pred}).to_csv(
            os.path.join(RESULTS_DIR, "fd001_chronos_full_history.csv"), index=False)

df_chr_full = pd.DataFrame(chronos_full_runs)
print(f"\nChronos-2 (FULL history) across seeds {SEEDS}:")
for m in ["mae", "rmse", "nasa"]:
    print(f"  {m.upper():>5}: {df_chr_full[m].mean():8.2f} ± {df_chr_full[m].std():5.2f} "
          f"(min {df_chr_full[m].min():.2f}, max {df_chr_full[m].max():.2f})")

df_chr_full["model"] = "Chronos_full"
df_chr_full.to_csv(os.path.join(RESULTS_DIR, "fd001_multiseed_chronos_full.csv"), index=False)
print("\nSaved: fd001_multiseed_chronos_full.csv + fd001_chronos_full_history.csv")

# Notebook 06 — Chronos-2: Complete Documentation

## Purpose

Train and evaluate the fourth and final model of the complexity ladder:
Chronos-2, a pretrained time-series foundation model, used with a frozen
backbone and a lightweight trained regression head. This is the only
model on the ladder that arrives with prior knowledge: it was pretrained
on large volumes of diverse time series before ever seeing FD001.

**Position in the project.** Fourth of four modeling notebooks
(03 Linear Regression → 04 XGBoost → 05 LSTM → 06 Chronos-2). Identical
frozen data, identical evaluation protocol, identical validation split
and tuning budget as XGBoost and LSTM.

**Why Chronos as step 4.** The LSTM (Notebook 05) demonstrated that
task-specific temporal learning nearly eliminates the dangerous
near-failure bias. Its remaining limitations: knowledge derived entirely
from 100 training engines of one simulated dataset, fragile training
dynamics (mean-collapse failure), GPU requirement, and no
interpretability. The foundation-model question: does broad, pretrained
temporal knowledge add anything on a simple single-condition task where
a task-trained specialist already performs strongly — and at what cost?

**Architecture (the published recipe).** Sensor window (last W cycles,
13 channels) → frozen Chronos-2 (native multivariate .embed(), 478 MB)
→ mean-pooled 768-dim embedding → small trained MLP head → RUL estimate.
Only the head trains; the backbone is never updated.

## Inputs and outputs

**Inputs (from `data/processed/`):** same three files as Notebook 05.

**Outputs:**
- `results/fd001_chronos_predictions.csv` — per-engine predictions
- `results/fd001_chronos_tuning_sweep.csv` — 12-row sweep table
- `results/fd001_chronos_extraction_times.csv` — cost accounting
- `figures/fd001_chronos_pred_vs_true.png`
- `figures/fd001_chronos_error_vs_true.png`
- Experiment log rows EXP-007 (linear), EXP-008 (piecewise125)

## Section overview

**Section 1 — Setup and variant verification.** chronos-forecasting
2.3.1 installed; Plan A (Chronos2Pipeline, native multivariate)
confirmed available alongside Plan B fallback (Chronos-T5). Lesson
recorded: a first Section 3 attempt ran 6+ hours incomplete on CPU
because the GPU runtime was not active — the same workload completes
in ~20 minutes on the T4. The GPU check in Section 1 exists precisely
to prevent this.

**Section 2 — Metrics + embedding pipeline.** Identical metrics as
Notebooks 03–05. API probe established the required input format:
3-d tensor (n_series, n_variates, history_length). Embedding output
per window: (13 variates, 6 tokens, 768 dims), mean-pooled to one
768-dim vector. Batched extraction throughput: ~6.6 ms/window on GPU.

**Section 3 — Extraction, sweep, final evaluation.** Same protocol:
20 validation engines (seed 42), truncation (seed 123), 6-config sweep
(context 30/50/100 × head (64,) / (128, 64)), selection on validation
RMSE, final head training on all 100 engines, single test evaluation
per label. Head training with label-scaled targets (the Notebook 05
lesson applied from the start), Adam lr=0.001, patience 10.

**Section 4 — Visualization, zone analysis, tail verification.**

**Section 5 — Save and finalize.**

## Sweep results (validation RMSE)

| Context | Head | Piecewise | Linear |
|---|---|---|---|
| 30 | (64,) | 23.59 | 29.23 |
| 50 | (64,) | 23.50 | 27.96 |
| 100 | (64,) | 19.52 | **17.94 ← selected** |
| 30 | (128, 64) | 23.65 | 30.58 |
| 50 | (128, 64) | 24.19 | 29.10 |
| 100 | (128, 64) | **18.78 ← selected** | 21.10 |

**Sweep finding — context length is decisive.** Validation RMSE improves
sharply from context 50 to 100 (piecewise: ~23.5 → ~19; linear:
~28 → ~18) while being nearly flat between 30 and 50. Both labels
independently selected the longest context. This confirms the
literature claim that longer contexts benefit foundation-model
embeddings, and contrasts with the LSTM, whose gains were between
15 and 50 cycles. The frozen model appears to extract additional value
from history beyond what the task-trained LSTM used.

## Results (recorded, final run)

| Run | Label | MAE | RMSE | NASA Score | Cost |
|---|---|---|---|---|---|
| EXP-007 | linear | 19.33 | 27.20 | 12341.71 | 4.1 min extraction + ~11 s head |
| EXP-008 | piecewise125 | 12.15 | 15.12 | 338.02 | 4.1 min extraction + 17.47 s head |

## Zone analysis — all four models (piecewise, mean bias)

| Zone | n | Chronos | LSTM | XGB | LR |
|---|---|---|---|---|---|
| Near failure (RUL ≤ 30) | 25 | +10.12 | +0.24 | +8.59 | +18.77 |
| Mid life (30 < RUL ≤ 80) | 20 | +8.81 | +5.58 | +8.83 | +19.75 |
| Early life (RUL > 80) | 55 | −4.64 | −2.34 | −3.21 | −8.46 |

Chronos is NOT the safest model on average near failure: its
near-failure bias (+10.12, with 23 of 25 near-failure engines predicted
late) is worse than XGBoost's and far from the LSTM's +0.24.

## The NASA-vs-MAE puzzle — verified mechanism

Chronos achieves the best NASA Score of the ladder (338.02) despite a
higher MAE than the LSTM. The verified explanation (per-engine NASA
contributions, Section 4f):

| | LSTM | Chronos |
|---|---|---|
| Total NASA | 402.13 | 338.02 |
| Max late error | +45.12 | +37.94 |
| Top-3 contributions | 90.1, 71.4, 19.4 | 43.4, 19.1, 17.1 |
| Top-3 share of total | 45% | 24% |

The LSTM's single worst prediction contributes 90.1 points — over a
quarter of its entire score. Chronos's worst contributes 43.4, and its
errors are more evenly spread. Because the NASA Score is exponential in
the error, it rewards Chronos's **shorter dangerous tail** over the
LSTM's better average. **Average accuracy and worst-case safety are
distinct axes — the LSTM wins the former, Chronos the latter.** This
reframes "which model is best?" as "best at what?" — a central point
for the Discussion chapter.

## Complexity-label interaction — fourth data point

Linear-label NASA Scores across the ladder: LR 14,094 → XGBoost 99,568
→ LSTM 23,147 → Chronos 12,342. Chronos with the linear label is the
least catastrophic of the flexible models — plausibly because only the
small head (not the representation) is trained on the flawed label.
Top-5 engines = 82.3% of its linear-label score; engine 67 is again the
worst offender (consistent with the LSTM's failure pattern). The
piecewise label remains essential for every model beyond LR.

## Strengths (Vorteile)

1. **Best NASA Score of the ladder (338.02)** via a shorter dangerous
   tail — no catastrophic single-engine failures.
2. **No backbone training.** The 478 MB model is used as delivered;
   only a small MLP head (seconds of training) adapts it to RUL.
   No mean-collapse-style fragility in the backbone.
3. **Native multivariate support.** Chronos-2 consumed all 13 sensor
   channels directly — no per-channel workarounds needed.
4. **Longer context exploited.** The only model whose selected input
   window (100 cycles) exceeded the LSTM's optimum (50), converting
   additional history into accuracy.
5. **Robustness to the flawed label** (relative): the least catastrophic
   linear-label NASA Score among the flexible models.

## Weaknesses (Nachteile)

1. **Does not beat the task-trained specialist on point accuracy.**
   MAE 12.15 vs LSTM 10.69; RMSE 15.12 vs 14.78. On a simple,
   single-condition dataset, pretrained general knowledge did not
   outperform task-specific training.
2. **Worst near-failure average bias of the sequence models** (+10.12) —
   23 of 25 near-failure engines predicted late. Operationally, its
   average behavior in the critical zone is inferior to the LSTM's.
3. **Highest total cost of the ladder.** ~4.1 min embedding extraction
   per context on GPU (12.3 min for the sweep's three contexts) plus
   head training — for headline metrics below an 81-second LSTM.
4. **Information loss in pooling.** Mean-pooling 13 variates × 6 tokens
   into one 768-dim vector discards channel identity; per-variate
   concatenation was deliberately avoided (overfitting risk at ~16.5k
   samples vs ~10k dims) and remains future work.
5. **Least transparent model.** A frozen pretrained transformer with a
   pooled embedding offers no feature-level interpretability.

## Ladder synthesis — the complete picture

| Model | MAE | RMSE | NASA | Near-fail bias | Cost |
|---|---|---|---|---|---|
| Linear Regression | 17.79 | 22.07 | 1334.57 | +18.77 | 0.02 s CPU |
| XGBoost (tuned) | 13.21 | 17.71 | 784.53 | +8.59 | 1.59 s CPU |
| LSTM (tuned) | **10.69** | **14.78** | 402.11 | **+0.24** | 80.74 s GPU |
| Chronos-2 + head | 12.15 | 15.12 | **338.02** | +10.12 | ~4.1 min + 17 s GPU |

**The ladder's answers to the research questions (data now complete):**

1. **Complexity pays — up to a point.** Each step from LR to LSTM bought
   clear accuracy gains at rising cost. The final step (LSTM → Chronos)
   cost the most and did not improve point accuracy: the complexity-cost
   curve flattens, and on MAE/RMSE reverses, at the top.
2. **Each step fixed a named weakness.** Non-linearity (XGBoost) halved
   the near-failure bias; temporal context (LSTM) eliminated it;
   pretrained breadth (Chronos) shortened the worst-case tail. The
   ladder is a reasoned progression, not model shopping.
3. **The label interacts with complexity.** The piecewise-125 label is
   not optional above LR; the linear label's flaws are amplified by
   every flexible model (NASA 12k–99k vs 338–1334 piecewise).
4. **"Best model" depends on the axis.** Average accuracy: LSTM.
   Worst-case safety (NASA): Chronos. Cost-efficiency: XGBoost
   (85% of the LSTM's MAE improvement at 2% of its wall-clock cost,
   on CPU). Interpretability: XGBoost. A deployment decision would
   weigh these axes, not read a single ranking.

**Modeling phase complete.** EXP-001 through EXP-008 recorded.
**Next:** Notebook 07 (comparison figures from the saved prediction
CSVs), then the writing phase: Chapters 6 (Methodology), 7 (Results),
2 (Theory), 3 (Related Work), 8 (Discussion), 9 (Conclusion).